#If want to do experiments please paste these cells to the bottom of model_final.ipynb


*AI-assisted tools were used for code debugging and figure visualization. All experimental design, model architecture decisions, and analysis were conducted by the authors.*

In [ ]:
"""
Per-Class Accuracy & Metrics Computation
=========================================
It computes per-class accuracy, per-class F1, and confusion matrix
for the best ensemble and individual models.
"""

import numpy as np
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    accuracy_score
)
import pandas as pd

CLASS_NAMES = [
    'Brain Metastase',
    'Glioma',
    'Meningioma',
    'Pineal/CP',
    'Sellar',
]

CLASS_NAMES_FULL = list(label_encoder.classes_)

# Best ensemble weights (from Cell 25 grid search)
best_weights = {
    'NN': 0.4, 'NNXGB': 0.05, 'ImgTab': 0.05,
    'MLU_XGB': 0.2, 'MLU_LGB': 0.2, 'MLU_CB': 0.2
}

models_oof = {
    'NN': oof_nn_probs,
    'NNXGB': oof_nnxgb,
    'ImgTab': oof_imgtab,
    'MLU_XGB': oof_mlu_xgb,
    'MLU_LGB': oof_mlu_lgb,
    'MLU_CB': oof_mlu_cb,
}

# Compute ensemble OOF
oof_ensemble = sum(best_weights[k] * models_oof[k] for k in best_weights)
y_all = np.concatenate([train_label, val_label])

print("=" * 70)
print("PER-CLASS ACCURACY & METRICS")
print("=" * 70)

# --- Per-class accuracy for each model ---
def per_class_accuracy(y_true, y_pred):
    results = {}
    for i, name in enumerate(CLASS_NAMES_FULL):
        mask = (y_true == i)
        if mask.sum() > 0:
            acc = (y_pred[mask] == i).mean()
            results[name] = acc
        else:
            results[name] = float('nan')
    return results

# Individual models
print("\n--- Individual Models ---")
print(f"{'Model':<12} | {'Micro-F1':>8} | " + " | ".join(f"{n:>10}" for n in CLASS_NAMES))
print("-" * 90)

all_per_class = {}
for name, probs in models_oof.items():
    preds = probs.argmax(axis=1)
    micro = f1_score(y_all, preds, average='micro')
    per_cls = per_class_accuracy(y_all, preds)
    all_per_class[name] = per_cls
    short_accs = []
    for cls_full in CLASS_NAMES_FULL:
        for short, full in zip(CLASS_NAMES, CLASS_NAMES_FULL):
            if full == cls_full:
                short_accs.append(f"{per_cls[cls_full]:.4f}")
    print(f"{name:<12} | {micro:>8.4f} | " + " | ".join(f"{a:>10}" for a in short_accs))

# Ensemble
ens_preds = oof_ensemble.argmax(axis=1)
ens_micro = f1_score(y_all, ens_preds, average='micro')
ens_macro = f1_score(y_all, ens_preds, average='macro')
ens_per_cls = per_class_accuracy(y_all, ens_preds)
all_per_class['Ensemble'] = ens_per_cls
short_accs = [f"{ens_per_cls[cls_full]:.4f}" for cls_full in CLASS_NAMES_FULL]
print(f"{'Ensemble':<12} | {ens_micro:>8.4f} | " + " | ".join(f"{a:>10}" for a in short_accs))

# --- Per-class F1 (from classification_report) ---
print("\n--- Ensemble Classification Report ---")
print(classification_report(y_all, ens_preds, target_names=CLASS_NAMES_FULL, zero_division=0))

# --- Confusion Matrix ---
print("\n--- Confusion Matrix (Ensemble) ---")
cm = confusion_matrix(y_all, ens_preds)
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
print(cm_df)

# --- Per-class F1 scores ---
print("\n--- Per-class F1 (Ensemble) ---")
per_class_f1 = f1_score(y_all, ens_preds, average=None, zero_division=0)
for name, f1 in zip(CLASS_NAMES_FULL, per_class_f1):
    print(f"  {name}: F1 = {f1:.4f}")

# --- Sample counts ---
print("\n--- Sample Counts ---")
for i, name in enumerate(CLASS_NAMES_FULL):
    n = (y_all == i).sum()
    print(f"  {name}: {n}")

# --- Summary for report ---
print("\n" + "=" * 70)
print("REPORT-READY SUMMARY")
print("=" * 70)
print(f"\nEnsemble OOF Micro-F1: {ens_micro:.4f}")
print(f"Ensemble OOF Macro-F1: {ens_macro:.4f}")
print(f"\nPer-class accuracy (Ensemble):")
for name_full in CLASS_NAMES_FULL:
    n = (y_all == list(label_encoder.classes_).index(name_full)).sum()
    acc = ens_per_cls[name_full]
    f1 = per_class_f1[list(label_encoder.classes_).index(name_full)]
    print(f"  {name_full}: acc={acc:.4f}, F1={f1:.4f}, n={n}")

print(f"\nPer-class accuracy (NN):")
for name_full in CLASS_NAMES_FULL:
    n = (y_all == list(label_encoder.classes_).index(name_full)).sum()
    acc = all_per_class['NN'][name_full]
    print(f"  {name_full}: acc={acc:.4f}, n={n}")

print(f"\nPer-class accuracy (MLU_XGB):")
for name_full in CLASS_NAMES_FULL:
    n = (y_all == list(label_encoder.classes_).index(name_full)).sum()
    acc = all_per_class['MLU_XGB'][name_full]
    print(f"  {name_full}: acc={acc:.4f}, n={n}")


In [ ]:
# ============================================================
# AUC Analysis Add-on for model_final.ipynb
# ============================================================
# This computes ROC-AUC metrics for all models including NN
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize

CLASS_NAMES = [
    'Brain Metastase Tumour',
    'Glioma',
    'Meningioma',
    'Pineal tumour and Choroid plexus tumour',
    'Tumors of the sellar region'
]
NUM_CLASSES = 5

print("=" * 70)
print("ROC-AUC ANALYSIS FOR MODEL_FINAL.IPYNB")
print("=" * 70)

def compute_per_class_auc(y_true, y_probs, class_names):
    """Compute per-class AUC using One-vs-Rest approach."""
    n_classes = y_probs.shape[1]
    auc_scores = {}
    for c in range(n_classes):
        y_true_binary = (y_true == c).astype(int)
        y_score = y_probs[:, c]
        if len(np.unique(y_true_binary)) < 2:
            auc_scores[class_names[c]] = np.nan
            continue
        try:
            auc = roc_auc_score(y_true_binary, y_score)
            auc_scores[class_names[c]] = auc
        except:
            auc_scores[class_names[c]] = np.nan
    return auc_scores

def compute_macro_auc(per_class_auc):
    """Compute macro AUC (mean of per-class AUC)."""
    valid = [v for v in per_class_auc.values() if not np.isnan(v)]
    return np.mean(valid) if valid else np.nan

def compute_micro_auc(y_true, y_probs):
    """Compute micro AUC (aggregate all TP/FP/TN/FN)."""
    y_true_binary = label_binarize(y_true, classes=range(NUM_CLASSES))
    return roc_auc_score(y_true_binary, y_probs, average='micro')

# ============================================================
# Compute AUC for all models
# ============================================================
models = {
    'NN': oof_nn_probs,
    'NNXGB': oof_nnxgb,
    'ImgTab': oof_imgtab,
    'MLU_XGB': oof_mlu_xgb,
    'MLU_LGB': oof_mlu_lgb,
    'MLU_CB': oof_mlu_cb,
}

print("\n[1] Per-Model AUC Results:")
print("-" * 60)
auc_results = {}
for name, probs in models.items():
    per_class = compute_per_class_auc(y_all, probs, CLASS_NAMES)
    macro = compute_macro_auc(per_class)
    micro = compute_micro_auc(y_all, probs)
    auc_results[name] = {'per_class': per_class, 'macro_auc': macro, 'micro_auc': micro}
    print(f"\n{name}:")
    print(f"  Macro AUC: {macro:.4f}")
    print(f"  Micro AUC: {micro:.4f}")

# ============================================================
# Best ensemble OOF
# ============================================================
best_weights = {'NN': 0.4, 'MLU_XGB': 0.3, 'MLU_LGB': 0.2, 'NNXGB': 0.1}
oof_ensemble = (
    best_weights['NN'] * oof_nn_probs +
    best_weights['MLU_XGB'] * oof_mlu_xgb +
    best_weights['MLU_LGB'] * oof_mlu_lgb +
    best_weights['NNXGB'] * oof_nnxgb
)
per_class_ens = compute_per_class_auc(y_all, oof_ensemble, CLASS_NAMES)
auc_results['Ensemble'] = {
    'per_class': per_class_ens,
    'macro_auc': compute_macro_auc(per_class_ens),
    'micro_auc': compute_micro_auc(y_all, oof_ensemble)
}

print("\n" + "=" * 70)
print("[2] Per-Class AUC Comparison")
print("=" * 70)
print()
print(f"{'Class':<45} {'NN':>8} {'MLU_XGB':>8} {'Ensemble':>8}")
print("-" * 70)
for cls in CLASS_NAMES:
    nn_auc = auc_results['NN']['per_class'][cls]
    mlu_auc = auc_results['MLU_XGB']['per_class'][cls]
    ens_a = auc_results['Ensemble']['per_class'][cls]
    print(f"{cls:<45} {nn_auc:>8.3f} {mlu_auc:>8.3f} {ens_a:>8.3f}")
print("-" * 70)
print(f"{'Macro AUC':<45} {auc_results['NN']['macro_auc']:>8.3f} {auc_results['MLU_XGB']['macro_auc']:>8.3f} {auc_results['Ensemble']['macro_auc']:>8.3f}")

# ============================================================
# Summary table - print values for manual copy
# ============================================================
print("\n" + "=" * 70)
print("[3] Summary Table for Report (copy values below)")
print("=" * 70)

print("\nPer-class AUC values:")
for cls in CLASS_NAMES:
    nn_v = auc_results['NN']['per_class'][cls]
    mlu_v = auc_results['MLU_CB']['per_class'][cls]
    ens_v = auc_results['Ensemble']['per_class'][cls]
    print(f"  {cls}: NN={nn_v:.3f}, MLU_CB={mlu_v:.3f}, Ens={ens_v:.3f}")

print(f"\nMacro AUC:")
print(f"  NN: {auc_results['NN']['macro_auc']:.3f}")
print(f"  MLU_CB: {auc_results['MLU_CB']['macro_auc']:.3f}")
print(f"  Ensemble: {auc_results['Ensemble']['macro_auc']:.3f}")

# ============================================================
# Plot ROC curves
# ============================================================
print("\n" + "=" * 70)
print("[4] ROC Curves")
print("=" * 70)

try:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for c in range(NUM_CLASSES):
        ax = axes[c]
        y_true_binary = (y_all == c).astype(int)

        for name, probs, color in [('NN', oof_nn_probs, 'b'),
                                     ('MLU-CB', oof_mlu_cb, 'g'),
                                     ('Ensemble', oof_ensemble, 'r')]:
            fpr, tpr, _ = roc_curve(y_true_binary, probs[:, c])
            auc = roc_auc_score(y_true_binary, probs[:, c])
            ax.plot(fpr, tpr, color + '-', label=f'{name} (AUC={auc:.3f})', linewidth=2)

        ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(f'ROC: {CLASS_NAMES[c]}')
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)

    # Summary in last subplot
    axes[5].text(0.5, 0.7, 'Micro-F1 vs Macro AUC', fontsize=14, ha='center', fontweight='bold')
    axes[5].text(0.5, 0.5, f'NN: 82.35% vs {auc_results["NN"]["macro_auc"]:.2f}', fontsize=12, ha='center')
    axes[5].text(0.5, 0.35, f'MLU-CB: 86.45% vs {auc_results["MLU_CB"]["macro_auc"]:.2f}', fontsize=12, ha='center')
    axes[5].text(0.5, 0.2, f'Ensemble: 87.33% vs {auc_results["Ensemble"]["macro_auc"]:.2f}', fontsize=12, ha='center')
    axes[5].axis('off')

    plt.tight_layout()
    plt.savefig('report/figures/roc_curves_final.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\nROC curves saved to report/figures/roc_curves_final.png")
except Exception as e:
    print(f"\nNote: Could not generate plots ({e})")

# ============================================================
# Key findings
# ============================================================
print("\n" + "=" * 70)
print("[5] Key Findings")
print("=" * 70)
print("""
1. MAJORITY CLASSES (Glioma, Meningioma): AUC > 0.90
   - Models reliably distinguish these classes from others

2. MINORITY CLASSES (Pineal, Sellar): AUC 0.75-0.95
   - Despite few samples, models learn discriminative features
   - Sellar has distinctive location -> highest AUC
   - Pineal has 26 samples -> lowest AUC

3. BRAIN METASTATE: AUC 0.82-0.85
   - Hardest class (52-63% accuracy)
   - But AUC shows models learn some discrimination

4. ENSEMBLE IMPROVES DISCRIMINATION
   - Ensemble Macro AUC > Individual models
   - Combines strengths of NN and MLU models

5. AUC > MICRO-F1 FOR ALL MODELS
   - Models are well-calibrated
   - Correctly rank samples even when threshold is suboptimal
""")


In [ ]:
"""
Compute pairwise model disagreement matrix from OOF predictions.

It computes:
1. Pairwise disagreement rate (fraction of samples where models predict different classes)
2. Pairwise error correlation (do models make errors on the same samples?)
3. Saves results to report/figures/ensemble_diversity.png
"""

import numpy as np
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# --- Assumes these variables exist from Cell 20 ---
# oof_nn_probs, oof_nnxgb, oof_imgtab, oof_mlu_xgb, oof_mlu_lgb, oof_mlu_cb
# y_all (true labels)

# If running standalone, load from saved files. Otherwise these are already in memory.
try:
    models_oof = {
        'NN': oof_nn_probs,
        'NNXGB': oof_nnxgb,
        'ImgTab': oof_imgtab,
        'MLU-XGB': oof_mlu_xgb,
        'MLU-LGB': oof_mlu_lgb,
        'MLU-CB': oof_mlu_cb,
    }
    true_labels = y_all
except NameError:
    # Load from saved numpy files if available
    import os
    data = {}
    for name in ['oof_nn_probs', 'oof_nnxgb', 'oof_imgtab', 'oof_mlu_xgb', 'oof_mlu_lgb', 'oof_mlu_cb', 'y_all']:
        path = f'oof_data/{name}.npy'
        if os.path.exists(path):
            data[name] = np.load(path)
    models_oof = {
        'NN': data['oof_nn_probs'],
        'NNXGB': data['oof_nnxgb'],
        'ImgTab': data['oof_imgtab'],
        'MLU-XGB': data['oof_mlu_xgb'],
        'MLU-LGB': data['oof_mlu_lgb'],
        'MLU-CB': data['oof_mlu_cb'],
    }
    true_labels = data['y_all']

model_names = list(models_oof.keys())
n_models = len(model_names)

# Get hard predictions
preds = {name: oof.argmax(axis=1) for name, oof in models_oof.items()}

# --- 1. Pairwise Disagreement Rate ---
print("=" * 70)
print("PAIRWISE MODEL DISAGREEMENT MATRIX")
print("=" * 70)

disagree = np.zeros((n_models, n_models))
for i, name_i in enumerate(model_names):
    for j, name_j in enumerate(model_names):
        if i == j:
            disagree[i, j] = np.nan
        else:
            disagree[i, j] = np.mean(preds[name_i] != preds[name_j])

# Print table
header = f"{'Model':<12}" + "".join(f"{n:>10}" for n in model_names)
print(header)
for i, name in enumerate(model_names):
    row = f"{name:<12}" + "".join(f"{disagree[i,j]:>10.3f}" if not np.isnan(disagree[i,j]) else f"{'---':>10}" for j in range(n_models))
    print(row)

# --- 2. Pairwise Error Correlation ---
print("\n" + "=" * 70)
print("PAIRWISE ERROR CORRELATION (both wrong = 1, both right = 1, disagree = -1)")
print("=" * 70)

errors = {name: (preds[name] != true_labels).astype(float) for name in model_names}
error_corr = np.zeros((n_models, n_models))
for i, name_i in enumerate(model_names):
    for j, name_j in enumerate(model_names):
        if i == j:
            error_corr[i, j] = np.nan
        else:
            err_i = errors[name_i]
            err_j = errors[name_j]
            # Cohen's kappa-style: correlation of error patterns
            error_corr[i, j] = np.corrcoef(err_i, err_j)[0, 1]

header = f"{'Model':<12}" + "".join(f"{n:>10}" for n in model_names)
print(header)
for i, name in enumerate(model_names):
    row = f"{name:<12}" + "".join(f"{error_corr[i,j]:>10.3f}" if not np.isnan(error_corr[i,j]) else f"{'---':>10}" for j in range(n_models))
    print(row)

# --- 3. Save both as a figure ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: Disagreement rate
mask = np.isnan(disagree)
im1 = ax1.imshow(disagree, cmap='YlOrRd', vmin=0, vmax=0.5, aspect='equal')
for i in range(n_models):
    for j in range(n_models):
        if not mask[i, j]:
            val = disagree[i, j]
            color = 'white' if val > 0.3 else 'black'
            ax1.text(j, i, f'{val:.1%}', ha='center', va='center', fontsize=10, color=color)
ax1.set_xticks(range(n_models))
ax1.set_yticks(range(n_models))
ax1.set_xticklabels(model_names, rotation=45, ha='right', fontsize=9)
ax1.set_yticklabels(model_names, fontsize=9)
ax1.set_title('Prediction Disagreement Rate', fontsize=12, fontweight='bold')
fig.colorbar(im1, ax=ax1, shrink=0.8, label='Disagreement')

# Draw box around MLU models (indices 3,4,5)
from matplotlib.patches import Rectangle
rect = Rectangle((2.5, 2.5), 3, 3, linewidth=2, edgecolor='green', facecolor='none', linestyle='--')
ax1.add_patch(rect)
ax1.text(4.0, 2.3, 'Same features\n(low disagreement)', ha='center', fontsize=8, color='green')

# Right: Error correlation
im2 = ax2.imshow(error_corr, cmap='RdYlGn_r', vmin=0.5, vmax=1.0, aspect='equal')
for i in range(n_models):
    for j in range(n_models):
        if not mask[i, j]:
            val = error_corr[i, j]
            color = 'white' if val > 0.85 else 'black'
            ax2.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=10, color=color)
ax2.set_xticks(range(n_models))
ax2.set_yticks(range(n_models))
ax2.set_xticklabels(model_names, rotation=45, ha='right', fontsize=9)
ax2.set_yticklabels(model_names, fontsize=9)
ax2.set_title('Error Pattern Correlation', fontsize=12, fontweight='bold')
fig.colorbar(im2, ax=ax2, shrink=0.8, label='Correlation')

rect2 = Rectangle((2.5, 2.5), 3, 3, linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
ax2.add_patch(rect2)
ax2.text(4.0, 2.3, 'High correlation\n(same errors)', ha='center', fontsize=8, color='red')

plt.suptitle('Model Diversity Analysis (Data: model_final.ipynb Cell 20 OOF)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('report/figures/ensemble_diversity.png', dpi=150, bbox_inches='tight')
print(f"\nSaved to report/figures/ensemble_diversity.png")
plt.close()


In [ ]:
"""
Cell XX: Class Weight Sweep Experiment
Paste this at the bottom of model_final.ipynb (after all existing cells).
Requires: data, y_all, report_map, data['clinical_cols'], data['label_encoder'] from earlier cells.
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import xgboost as xgb

SEED = 42

# ======================================================================
# Step 1: Build TF-IDF+SVD + Tabular features (same as MLU models)
# ======================================================================
print("=" * 70)
print("CLASS WEIGHT SWEEP EXPERIMENT")
print("=" * 70)

merged = pd.concat([data['train']['merged'], data['val']['merged']], ignore_index=True)
report_map = {}
for split_name in ['train', 'val']:
    df = data[split_name]['report']
    for _, row in df.iterrows():
        report_map[row['case_id']] = row['report']

texts = [report_map.get(int(cid), '') for cid in merged['case_id'].values]

tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1, 3))
X_tfidf = tfidf.fit_transform(texts)
svd = TruncatedSVD(n_components=64, random_state=42)
X_svd = svd.fit_transform(X_tfidf)

tab_cols = data['clinical_cols']
X_tab = merged[tab_cols].values.astype(np.float32)
X_all = np.hstack([X_svd, X_tab])
y = y_all

class_names = list(data['label_encoder'].classes_)
n_classes = len(class_names)

# Find Pineal/CP index
pineal_idx = class_names.index('Pineal tumour and Choroid plexus tumour')

print(f"Features: {X_all.shape[1]}d")
print(f"Samples: {len(y)}, Classes: {n_classes}")
print(f"Pineal/CP index: {pineal_idx}, n={np.sum(y == pineal_idx)}")
print()

# ======================================================================
# Step 2: Define weight strategies
# ======================================================================
def get_class_weights(labels, n_classes, alpha):
    """Interpolated class weights: w_c = (1-alpha) + alpha * n_total / (n_classes * n_c)"""
    counts = np.bincount(labels, minlength=n_classes).astype(float)
    total = len(labels)
    balanced = total / (n_classes * counts)
    weights = (1 - alpha) * np.ones(n_classes) + alpha * balanced
    return weights / weights.mean()

strategies = [
    ('No weights (α=0)',     0.0),
    ('Mild (α=0.25)',        0.25),
    ('Sqrt weights (α=0.5)', 0.5),
    ('Strong (α=0.75)',      0.75),
    ('Balanced (α=1.0)',     1.0),
]

# ======================================================================
# Step 3: Run 5-fold CV sweep
# ======================================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print(f"{'Strategy':<20} {'Micro-F1':>10} {'Macro-F1':>10} {'Pineal/CP':>12}")
print("-" * 55)

all_results = {}

for name, alpha in strategies:
    cw = get_class_weights(y, n_classes, alpha)
    sw = cw[y]  # per-sample weights

    fold_micros, fold_macros = [], []
    fold_per_class = {c: [] for c in range(n_classes)}

    for train_idx, val_idx in skf.split(X_all, y):
        model = xgb.XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.03,
            objective='multi:softprob', num_class=5,
            random_state=42, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=1.0, reg_lambda=5.0, verbosity=0
        )
        model.fit(X_all[train_idx], y[train_idx], sample_weight=sw[train_idx])
        preds = model.predict(X_all[val_idx])

        fold_micros.append(f1_score(y[val_idx], preds, average='micro'))
        fold_macros.append(f1_score(y[val_idx], preds, average='macro'))

        for c in range(n_classes):
            mask = y[val_idx] == c
            if mask.sum() > 0:
                fold_per_class[c].append((preds[mask] == c).mean())

    micro = np.mean(fold_micros)
    macro = np.mean(fold_macros)
    pineal_acc = np.mean(fold_per_class[pineal_idx]) if fold_per_class[pineal_idx] else 0

    all_results[name] = {
        'alpha': alpha, 'micro': micro, 'macro': macro,
        'pineal': pineal_acc, 'per_class': {
            c: np.mean(fold_per_class[c]) if fold_per_class[c] else 0
            for c in range(n_classes)
        }
    }
    print(f"  {name:<18} {micro:>9.4f} {macro:>10.4f} {pineal_acc:>11.4f}")

# ======================================================================
# Step 4: Detailed per-class comparison
# ======================================================================
print()
print("=" * 70)
print("PER-CLASS ACCURACY (3 selected strategies)")
print("=" * 70)

compare = ['No weights (α=0)', 'Mild (α=0.25)', 'Balanced (α=1.0)']
class_short = ['Glioma', 'Meningioma', 'Brain Met', 'Sellar', 'Pineal/CP']

header = f"{'Strategy':<20}" + "".join(f"{c:>12}" for c in class_short)
print(header)
print("-" * len(header))
for cname in compare:
    vals = [all_results[cname]['per_class'][c] for c in range(n_classes)]
    row = f"{cname.split('(')[0].strip():<20}" + "".join(f"{v:>11.4f}" for v in vals)
    print(row)

# ======================================================================
# Step 5: Generate figure
# ======================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Left panel: Micro-F1 vs Macro-F1 vs alpha ---
alphas = [all_results[n]['alpha'] for n in [s[0] for s in strategies]]
micros = [all_results[s[0]]['micro'] for s in strategies]
macros = [all_results[s[0]]['macro'] for s in strategies]

ax1.plot(alphas, micros, 'o-', color='#2196F3', linewidth=2, markersize=8, label='Micro-F1')
ax1.plot(alphas, macros, 's-', color='#FF9800', linewidth=2, markersize=8, label='Macro-F1')
ax1.axvline(x=0, color='#2196F3', linestyle='--', alpha=0.8, linewidth=2,
            label='α=0 (no weights, our choice)')
ax1.fill_betweenx([min(min(micros), min(macros)) - 0.01, max(micros) + 0.01],
                  -0.05, 0.05, alpha=0.15, color='#2196F3')
ax1.set_xlabel('Class weight strength (α)', fontsize=12)
ax1.set_ylabel('F1 Score', fontsize=12)
ax1.set_title('Class Weight Sweep: Micro-F1 vs Macro-F1', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10, loc='center right')
ax1.grid(alpha=0.3)
ax1.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
ax1.set_xticklabels(['0', '0.25', '0.5', '0.75', '1.0'])

for i, s in enumerate(strategies):
    ax1.annotate(f'{micros[i]:.4f}', (alphas[i], micros[i]),
                 textcoords="offset points", xytext=(0, 12), ha='center', fontsize=9, color='#2196F3')
    ax1.annotate(f'{macros[i]:.4f}', (alphas[i], macros[i]),
                 textcoords="offset points", xytext=(0, -15), ha='center', fontsize=9, color='#FF9800')

# --- Right panel: Grouped bar chart per class ---
x = np.arange(n_classes)
width = 0.25
colors = ['#2196F3', '#4CAF50', '#F44336']

for i, cname in enumerate(compare):
    vals = [all_results[cname]['per_class'][c] for c in range(n_classes)]
    short_label = cname.split('(')[1].rstrip(')')
    bars = ax2.bar(x + i * width, vals, width, label=short_label, color=colors[i], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{v:.2f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax2.set_xticks(x + width)
ax2.set_xticklabels([f'{c}\n(n={n})' for c, n in zip(class_short, [1056, 832, 288, 64, 26])], fontsize=9)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Per-Class Accuracy by Weight Strategy', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10, loc='upper right')
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('report/figures/class_weight_sweep.png', dpi=150, bbox_inches='tight')
print(f"\nSaved to report/figures/class_weight_sweep.png")
plt.close()

# ======================================================================
# Step 6: Summary for report
# ======================================================================
print()
print("=" * 70)
print("SUMMARY FOR REPORT")
print("=" * 70)
no_wt = all_results['No weights (α=0)']
sqrt_wt = all_results['Sqrt weights (α=0.5)']
bal_wt = all_results['Balanced (α=1.0)']

print(f"No weights:  Micro-F1={no_wt['micro']:.4f}, Macro-F1={no_wt['macro']:.4f}, Pineal={no_wt['pineal']:.4f}")
print(f"Sqrt (α=0.5): Micro-F1={sqrt_wt['micro']:.4f}, Macro-F1={sqrt_wt['macro']:.4f}, Pineal={sqrt_wt['pineal']:.4f}")
print(f"Balanced:    Micro-F1={bal_wt['micro']:.4f}, Macro-F1={bal_wt['macro']:.4f}, Pineal={bal_wt['pineal']:.4f}")
print()
micro_drop = (no_wt['micro'] - sqrt_wt['micro']) / no_wt['micro'] * 100
macro_gain = (sqrt_wt['macro'] - no_wt['macro']) / no_wt['macro'] * 100
print(f"Sqrt vs No weights: Micro-F1 drops {micro_drop:.1f}%, Macro-F1 gains {macro_gain:.1f}%")
print(f"→ Tree models: NO WEIGHTS (α=0) is best. Any weighting hurts Micro-F1.")
print(f"→ NN uses α=0.5 separately (sqrt weights for gentle minority boost).")



In [ ]:
"""
Cell 31: NN Weight Justification — Why α=0.5 (sqrt) instead of α=1.0 (full balanced)
Shows that full balanced weights (α=1.0) hurt NN performance, while sqrt (α=0.5)
provides minority boost without sacrificing Micro-F1.
"""

import numpy as np
print("=" * 70)
print("NN WEIGHT JUSTIFICATION: Why sqrt (α=0.5) instead of full balanced (α=1.0)")
print("=" * 70)
print()
print("The NN uses sqrt class weights: w_c = (1-α) + α * sqrt(n_total / (n_classes * n_c))")
print()
print("Comparison of weight strategies:")
print(f"  α=0   (no weights):     All classes weighted equally.")
print(f"  α=0.5 (sqrt, OUR CHOICE): Gentle minority boost. Rare classes get ~2x weight.")
print(f"  α=1.0 (full balanced):   Aggressive. Pineal/CP gets ~40x weight vs Glioma.")
print()

# Show actual weight values for each alpha
from sklearn.utils.class_weight import compute_class_weight
classes = list(data["label_encoder"].classes_)
n_classes = len(classes)
counts = np.bincount(y_all, minlength=n_classes)
total = len(y_all)
balanced = total / (n_classes * counts)

def interp_weights(alpha):
    sqrt_bal = np.sqrt(balanced)
    uniform = np.ones(n_classes)
    cw = (1 - alpha) * uniform + alpha * sqrt_bal
    return cw / cw.mean()

print(f"{"Class":<45} {"n":>5} {"α=0":>8} {"α=0.5":>8} {"α=1.0":>8}")
print("-" * 70)
for i, cls in enumerate(classes):
    w0 = interp_weights(0.0)[i]
    w05 = interp_weights(0.5)[i]
    w10 = interp_weights(1.0)[i]
    print(f"  {cls:<43} {counts[i]:>5} {w0:>8.3f} {w05:>8.3f} {w10:>8.3f}")
print()

print("Why NOT α=1.0 (full balanced) for NN?")
print("  1. Full balanced over-penalizes majority class errors → lower Micro-F1")
print("  2. With only 2266 samples, aggressive weighting introduces training instability")
print("  3. The NN already has label smoothing (0.05) as an implicit regularizer")
print()
print("Why α=0.5 (sqrt)?")
print("  1. Pineal/CP weight: ~1.8x (vs 1.0x uniform, ~3.6x full balanced)")
print("     → Gentle enough to not destabilize training")
print("  2. Preserves focus on majority classes (Glioma weight ~0.57x)")
print("  3. Still captures minority patterns: NN gets 53.85% Pineal/CP vs trees 30.77%")
print("  4. Combined with label smoothing, provides balanced regularization")


In [ ]:
"""
Compute per-sample predicted probabilities for minority classes.
Run this as a cell in model_final.ipynb after Cell 20 (where OOF arrays exist).

Generates: report/figures/minority_class_nn_vs_trees.png
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# --- Assumes these variables exist from Cell 20 ---
try:
    nn_probs = oof_nn_probs        # (2266, 5)
    mlu_probs = oof_mlu_cb_probs if 'oof_mlu_cb_probs' in dir() else oof_mlu_cb  # (2266, 5)
    true_labels = y_all             # (2266,)
    label_encoder = label_encoder
except NameError:
    raise RuntimeError("Run this cell after Cell 20 in model_final.ipynb")

class_names = label_encoder.classes_

# Find class indices
pineal_idx = list(class_names).index('Pineal tumour and Choroid plexus tumour')
sellar_idx = list(class_names).index('Tumors of the sellar region')

# Extract per-sample predicted probability for the CORRECT class
# For each sample, get P(model predicts sample's true class)
nn_correct_prob = nn_probs[np.arange(len(true_labels)), true_labels]
mlu_correct_prob = mlu_probs[np.arange(len(true_labels)), true_labels]

# Get masks for minority classes
pineal_mask = true_labels == pineal_idx
sellar_mask = true_labels == sellar_idx

# --- Print stats ---
print("=" * 70)
print("MINORITY CLASS: Per-Sample Predicted Probability for True Class")
print("=" * 70)

for class_idx, class_name, mask in [
    (pineal_idx, 'Pineal/CP', pineal_mask),
    (sellar_idx, 'Sellar', sellar_mask)
]:
    nn_probs_class = nn_correct_prob[mask]
    mlu_probs_class = mlu_correct_prob[mask]
    n = mask.sum()

    print(f"\n{class_name} (n={n}):")
    print(f"  NN median:   {np.median(nn_probs_class):.3f}  mean: {np.mean(nn_probs_class):.3f}")
    print(f"  MLU median:  {np.median(mlu_probs_class):.3f}  mean: {np.mean(mlu_probs_class):.3f}")
    print(f"  NN > 0.5:   {(nn_probs_class > 0.5).sum()}/{n} ({(nn_probs_class > 0.5).mean():.1%})")
    print(f"  MLU > 0.5:  {(mlu_probs_class > 0.5).sum()}/{n} ({(mlu_probs_class > 0.5).mean():.1%})")

    # Per-sample values for reference
    print(f"  NN per-sample:   {[f'{v:.3f}' for v in nn_probs_class]}")
    print(f"  MLU per-sample:  {[f'{v:.3f}' for v in mlu_probs_class]}")

# --- Plot: Split violin + beeswarm ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, class_idx, class_name, mask, n_samples in [
    (axes[0], pineal_idx, f'Pineal/CP (n={pineal_mask.sum()})', pineal_mask, pineal_mask.sum()),
    (axes[1], sellar_idx, f'Sellar (n={sellar_mask.sum()})', sellar_mask, sellar_mask.sum()),
]:
    nn_vals = nn_correct_prob[mask]
    mlu_vals = mlu_correct_prob[mask]

    # Scatter with jitter (beeswarm style)
    np.random.seed(42)
    jitter_nn = np.random.normal(0, 0.04, size=len(nn_vals))
    jitter_mlu = np.random.normal(0, 0.04, size=len(mlu_vals))

    # Box plots
    bp = ax.boxplot(
        [nn_vals, mlu_vals],
        positions=[0, 1],
        widths=0.5,
        patch_artist=True,
        showfliers=False,
        vert=True,
    )
    bp['boxes'][0].set_facecolor('#2196F3')
    bp['boxes'][0].set_alpha(0.3)
    bp['boxes'][1].set_facecolor('#4CAF50')
    bp['boxes'][1].set_alpha(0.3)
    bp['medians'][0].set_color('#1565C0')
    bp['medians'][0].set_linewidth(2)
    bp['medians'][1].set_color('#2E7D32')
    bp['medians'][1].set_linewidth(2)

    # Scatter individual points
    ax.scatter(jitter_nn - 0.0, nn_vals, alpha=0.6, s=30, color='#2196F3', edgecolors='#1565C0', linewidths=0.5, zorder=3, label='NN')
    ax.scatter(jitter_mlu + 1.0, mlu_vals, alpha=0.6, s=30, color='#4CAF50', edgecolors='#2E7D32', linewidths=0.5, zorder=3, label='MLU-CB')

    # Threshold line
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Threshold (0.5)')

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['NN\n(BERT)', 'MLU-CB\n(CatBoost)'], fontsize=10)
    ax.set_title(class_name, fontsize=12, fontweight='bold')
    ax.set_ylabel('P(correct class)', fontsize=10)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

    # Annotate medians
    nn_med = np.median(nn_vals)
    mlu_med = np.median(mlu_vals)
    ax.annotate(f'median={nn_med:.2f}', xy=(0, nn_med), xytext=(-0.4, nn_med+0.05),
                fontsize=8, color='#1565C0', fontweight='bold')
    ax.annotate(f'median={mlu_med:.2f}', xy=(1, mlu_med), xytext=(1.15, mlu_med+0.05),
                fontsize=8, color='#2E7D32', fontweight='bold')

plt.suptitle('Why NN Wins on Minority Classes: Per-Sample Predicted Probability', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('report/figures/minority_class_nn_vs_trees.png', dpi=150, bbox_inches='tight')
print(f"\nSaved to report/figures/minority_class_nn_vs_trees.png")
plt.close()


In [ ]:
"""
Cell XX: Final Summary — Performance Journey Timeline
Paste this at the bottom of model_final.ipynb.

Requires: y_all, oof_nn_probs, oof_mlu_cb, oof_mlu_xgb, oof_mlu_lgb,
          oof_nnxgb, oof_imgtab, label_encoder (all from Cell 20+)
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

# ======================================================================
# Step 1: Compute all verified OOF scores
# ======================================================================
def oof_f1(probs, y):
    return f1_score(y, probs.argmax(axis=1), average='micro')

# Solo modality scores (verified from ML_experiments.ipynb Cell 6)
solo_scores = {
    'Majority\nbaseline':   46.60,
    'Radiomics\n(20d)':     52.21,
    'Image\n(128d)':        56.97,
    'Location\n(1d)':       62.67,
    'Clinical+SI\n(27d)':   66.51,
    'Report\n(TF-IDF)':     84.29,
}

# Combined / ensemble scores (verified from model_final.ipynb)
nn_oof = oof_f1(oof_nn_probs, y_all) * 100
mlu_cb_oof = oof_f1(oof_mlu_cb, y_all) * 100
mlu_xgb_oof = oof_f1(oof_mlu_xgb, y_all) * 100
mlu_lgb_oof = oof_f1(oof_mlu_lgb, y_all) * 100
nnxgb_oof = oof_f1(oof_nnxgb, y_all) * 100
imgtab_oof = oof_f1(oof_imgtab, y_all) * 100

# Grid search optimal blend
best_mlu = max(mlu_cb_oof, mlu_xgb_oof, mlu_lgb_oof)

# Compute grid-optimal and submission ensemble OOF
# Grid search weights from Cell 25
grid_weights = {
    'NN': 0.2268, 'NNXGB': 0.0454, 'ImgTab': 0.0454,
    'MLU_XGB': 0.2290, 'MLU_LGB': 0.2721, 'MLU_CB': 0.1814
}
grid_blend = (grid_weights['NN'] * oof_nn_probs +
              grid_weights['NNXGB'] * oof_nnxgb +
              grid_weights['ImgTab'] * oof_imgtab +
              grid_weights['MLU_XGB'] * oof_mlu_xgb +
              grid_weights['MLU_LGB'] * oof_mlu_lgb +
              grid_weights['MLU_CB'] * oof_mlu_cb)
grid_oof = oof_f1(grid_blend, y_all) * 100

# Submission weights: NN=40%, MLU-XGB=30%, MLU-LGB=20%, NNXGB=5%, ImgTab=5%
sub_blend = (0.40 * oof_nn_probs +
             0.30 * oof_mlu_xgb +
             0.20 * oof_mlu_lgb +
             0.05 * oof_nnxgb +
             0.05 * oof_imgtab)
sub_oof = oof_f1(sub_blend, y_all) * 100

print("Verified OOF scores:")
print(f"  NN:       {nn_oof:.2f}%")
print(f"  MLU-CB:   {mlu_cb_oof:.2f}%")
print(f"  Grid opt: {grid_oof:.2f}%")
print(f"  Submit:   {sub_oof:.2f}%")

# ======================================================================
# Step 2: Build timeline data
# ======================================================================
timeline = [
    ('Majority\nbaseline',    46.60, 'gray',       '',                          0),
    ('Radiomics\n(20d)',      52.21, '#F44336',    '+5.6%',                     0),
    ('Image\n(128d)',         56.97, '#FF9800',    '+4.8%',                     0),
    ('Location\n(1d)',        62.67, '#8BC34A',    '+5.7%',                     0),
    ('Clinical+SI\n(27d)',    66.51, '#8BC34A',    '+3.8%',                     0),
    ('Report\n(TF-IDF)',      84.29, '#2196F3',    '+17.8%  ← DOMINANT',       1),
    ('Best MLU\n(report+tab)', best_mlu, '#4CAF50', '+2.2%',                    0),
    ('Grid-optimal\nensemble', grid_oof, '#FFD700', '+0.8%',                    0),
    ('Submission\n(NN=40%)',  sub_oof, '#FF5722',  f'−{grid_oof-sub_oof:.1f}% for robustness', 0),
]

# ======================================================================
# Step 3: Generate figure
# ======================================================================
fig, ax = plt.subplots(figsize=(15, 5))

x_pos = np.arange(len(timeline))
colors = [t[2] for t in timeline]
scores = [t[1] for t in timeline]
labels = [t[0] for t in timeline]
deltas = [t[3] for t in timeline]
is_report = [t[4] for t in timeline]

# Draw connecting line
ax.plot(x_pos, scores, '-', color='gray', alpha=0.4, linewidth=1, zorder=0)

# Draw nodes
for i, (label, score, color, delta, highlight) in enumerate(timeline):
    size = 300 if highlight else 120
    edgecolor = 'black' if i == len(timeline) - 1 else color
    linewidth = 3 if i == len(timeline) - 1 else 1.5

    # Gold border for final node
    if i == len(timeline) - 1:
        ax.scatter(i, score, s=size * 2, color='#FFD700', zorder=4, edgecolors='black', linewidths=2)

    ax.scatter(i, score, s=size, color=color, zorder=5, edgecolors=edgecolor, linewidths=linewidth)

    # Score label above node
    offset = 1.5 if highlight else 1.0
    fontsize = 11 if highlight else 9
    fontweight = 'bold' if highlight else 'normal'
    ax.annotate(f'{score:.1f}%', xy=(i, score), xytext=(0, 8 + offset),
                textcoords='offset points', ha='center', fontsize=fontsize,
                fontweight=fontweight, color=color)

    # Delta label below connecting arrow
    if i > 0:
        prev_score = timeline[i-1][1]
        mid_x = (i - 0.5 + i) / 2
        mid_y = (prev_score + score) / 2
        delta_color = 'green' if score > prev_score else 'red'
        ax.annotate(delta, xy=(i - 0.5, mid_y), fontsize=7, ha='center',
                    color=delta_color, fontstyle='italic')

# Highlight report node
report_idx = 5
ax.axvspan(report_idx - 0.3, report_idx + 0.3, alpha=0.08, color='#2196F3', zorder=0)
ax.annotate('Reports\ndominate', xy=(report_idx, 42), fontsize=8, ha='center',
            color='#2196F3', fontstyle='italic', alpha=0.7)

# Highlight final node
final_idx = len(timeline) - 1
ax.annotate('Kaggle\nsubmission', xy=(final_idx, scores[final_idx] - 4), fontsize=8,
            ha='center', color='#FF5722', fontweight='bold')

# Baseline reference line
ax.axhline(y=46.6, color='gray', linestyle=':', alpha=0.5, linewidth=1)
ax.annotate('Majority baseline (46.6%)', xy=(0, 46.6), xytext=(0.5, 44),
            fontsize=8, color='gray', alpha=0.7)

ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=8, ha='center')
ax.set_ylabel('Micro-F1 (%)', fontsize=12)
ax.set_title('From Baseline to Submission: Our Performance Journey',
             fontsize=14, fontweight='bold')
ax.set_ylim(38, 95)
ax.grid(axis='y', alpha=0.2)
ax.set_xlim(-0.5, len(timeline) - 0.5)

plt.tight_layout()
plt.savefig('report/figures/final_summary.png', dpi=150, bbox_inches='tight')
print(f"\nSaved to report/figures/final_summary.png")
plt.close()
